In [0]:
# Detecta el catálogo actual (suele ser 'workspace' o 'main')
catalog = spark.sql("select current_catalog()").first()[0]

schema = "raw"
volume = "files"
volume_ouput ="files_output"

In [0]:
# Crea esquema y volumen si no existen
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume_ouput}")
display(spark.sql("SHOW VOLUMES IN workspace.raw"))

In [0]:

# Elimina todos los archivos y subcarpetas dentro de la ruta
dbutils.fs.rm("/Volumes/workspace/raw/files_output", recurse=True)

In [0]:
%pip install -q azure-ai-documentintelligence azure-identity
dbutils.library.restartPython()

In [0]:
TENANT_ID     = ""
CLIENT_ID     = ""
CLIENT_SECRET = dbutils.secrets.get(scope="client-secrets", key="document-intelligence-secret")
ENDPOINT      = "https://ocr-text-extractor-v1.cognitiveservices.azure.com/"

# Volúmenes
VOL_ENTRADA = "/Volumes/workspace/raw/files"
VOL_SALIDA  = "/Volumes/workspace/raw/files_output"

In [0]:
from azure.identity import ClientSecretCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient

credential = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
client = DocumentIntelligenceClient(ENDPOINT, credential)

print("Conectado a", ENDPOINT)

In [0]:
import datetime as dt

# El scope es el del recurso al que llamas (Azure AI / Cognitive Services)
token = credential.get_token("https://cognitiveservices.azure.com/.default")

print(token.token)


In [0]:
import base64, json

payload = token.token.split(".")[1]
payload += "=" * (-len(payload) % 4)          # padding que Azure omite
claims = json.loads(base64.urlsafe_b64decode(payload))

for k in ("aud", "iss", "tid", "appid", "app_displayname", "roles", "exp"):
    if k in claims:
        print(f"{k:18} {claims[k]}")

In [0]:
from pathlib import Path

pdfs = sorted(Path(VOL_ENTRADA).glob("*.pdf"))

for pdf in pdfs:
    print(f"{pdf.name:<25} {pdf.stat().st_size / 1024:>8.1f} KB")

print(f"\nTotal: {len(pdfs)} PDF(s)")

In [0]:
from azure.identity import ClientSecretCredential
import base64, json

cred = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
token = cred.get_token("https://cognitiveservices.azure.com/.default")

payload = token.token.split(".")[1]
payload += "=" * (-len(payload) % 4)
claims = json.loads(base64.b64decode(payload))
print(claims["appid"], claims["aud"])

In [0]:
import json

Path(VOL_SALIDA).mkdir(parents=True, exist_ok=True)

for pdf in pdfs:
    # 1. Enviar el PDF a Document Intelligence
    with open(pdf, "rb") as f:
        poller = client.begin_analyze_document(
            "prebuilt-read", body=f, content_type="application/octet-stream"
        )
    resultado = poller.result()

    # 2. Armar el JSON
    datos = {
        "archivo": pdf.name,
        "paginas": len(resultado.pages),
        "texto": resultado.content,
        "texto_por_pagina": [
            {
                "pagina": p.page_number,
                "texto": "\n".join(linea.content for linea in p.lines),
            }
            for p in resultado.pages
        ],
    }

    # 3. Escribirlo en el volumen de salida
    destino = Path(VOL_SALIDA) / f"{pdf.stem}.json"
    with open(destino, "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)

    print(f"OK  {pdf.name:<25} -> {destino.name:<25} "
          f"{datos['paginas']} pág, {len(datos['texto'])} caracteres")

print("\nListo.")

In [0]:
for archivo in sorted(Path(VOL_SALIDA).glob("*.json")):
    print(f"{archivo.name:<25} {archivo.stat().st_size / 1024:>8.1f} KB")

In [0]:
# Abrir uno para ver cómo quedó
with open(Path(VOL_SALIDA) / f"{pdfs[0].stem}.json", encoding="utf-8") as f:
    ejemplo = json.load(f)

print("Archivo:", ejemplo["archivo"])
print("Páginas:", ejemplo["paginas"])
print("\n--- Texto de la página 1 ---\n")
print(ejemplo["texto_por_pagina"][0]["texto"][:800])